# NF Sweep Inspection

This notebook is ordered around the validation checks that matter first:

1. discover runs and samples
2. load real and generated arrays without blowing up memory
3. one-point statistics
4. power spectra P(k)
5. image grids
6. PCA encoder diagnostics and PCA-based generalizability
7. training/EMA curves as supporting context

Set `SWEEP_NAME = "nf_sweep"` for the original sweep or `SWEEP_NAME = "nf_sweep_v2"` for Nick's merged-main sweep.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", Path.cwd())).resolve()
if not (PROJECT_DIR / "scripts").exists():
    PROJECT_DIR = Path("/home/jiamingp/diffusion_models_repo")

SWEEP_NAME = os.environ.get("SWEEP_NAME", "nf_sweep")  # "nf_sweep" or "nf_sweep_v2"
DEFAULT_COSMODIFF_DIR = {
    "nf_sweep": "/home/jiamingp/Diffusion_model/cosmo_diffusion_normalization_fixes_git",
    "nf_sweep_v2": "/home/jiamingp/Diffusion_model/cosmo_diffusion_main",
}.get(SWEEP_NAME, "/home/jiamingp/Diffusion_model/cosmo_diffusion_main")
COSMODIFF_DIR = Path(os.environ.get("COSMODIFF_DIR_OVERRIDE", DEFAULT_COSMODIFF_DIR)).resolve()

if COSMODIFF_DIR.exists() and str(COSMODIFF_DIR) not in sys.path:
    sys.path.insert(0, str(COSMODIFF_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})

print("project:", PROJECT_DIR)
print("sweep:", SWEEP_NAME)
print("cosmodiff_dir:", COSMODIFF_DIR, "exists=", COSMODIFF_DIR.exists())

In [ ]:
CONFIG_DIR = PROJECT_DIR / "local" / SWEEP_NAME / "configs"
MANIFEST_PATH = PROJECT_DIR / "local" / SWEEP_NAME / "manifest.json"
CHECKPOINT_ROOT = Path(f"/scratch/huterer_root/huterer0/jiamingp/saved_runs/{SWEEP_NAME}")
SAMPLE_ROOT = PROJECT_DIR / "results" / SWEEP_NAME / "samples"
OUTPUT_DIR = PROJECT_DIR / "results" / SWEEP_NAME / "inspection"
CACHE_DIR = PROJECT_DIR / "results" / "cache" / f"{SWEEP_NAME}_inspection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 123
if SWEEP_NAME == "nf_sweep_v2":
    EMA_LABELS = ["raw", "ema0p02", "ema0p04", "ema0p06", "ema0p08", "ema0p10"]
    EMA_VALUES = {"raw": np.nan, "ema0p02": 0.02, "ema0p04": 0.04, "ema0p06": 0.06, "ema0p08": 0.08, "ema0p10": 0.10}
    PREFERRED_SAMPLE_LABEL = "ema0p06"
    PREFERRED_SAMPLER = "train_full"
else:
    EMA_LABELS = ["raw", "ema0p03", "ema0p05", "ema0p08", "ema0p13", "ema0p20"]
    EMA_VALUES = {"raw": np.nan, "ema0p03": 0.03, "ema0p05": 0.05, "ema0p08": 0.08, "ema0p13": 0.13, "ema0p20": 0.20}
    PREFERRED_SAMPLE_LABEL = "ema0p13"
    PREFERRED_SAMPLER = None

# Keep these modest on login nodes. 16 raw cubes = 512 2D slices for zthin=4.
# Raise on a compute node if you want smaller statistical error bars.
MAX_RAW_REAL_CUBES = 16
MAX_REAL_HIST = 512
MAX_REAL_PK = 512
MAX_REAL_IMAGES = 8
MAX_GENERATED_HIST = 128
MAX_GENERATED_PK = 128
MAX_GENERATED_IMAGES = 8
PK_NBINS = 25

# PCA encoder settings. PCA is fit on the real reference and then applied to real/generated.
PCA_N_COMPONENTS = 32
PCA_MAX_FIT_REAL = 512
PCA_MAX_REAL = 512
PCA_MAX_GENERATED = 128
PCA_COPY_QUANTILE = 0.99  # threshold = 99th percentile of real-to-real nearest-neighbor cosine similarity

print("manifest:", MANIFEST_PATH, "exists=", MANIFEST_PATH.exists())
print("checkpoint_root:", CHECKPOINT_ROOT)
print("sample_root:", SAMPLE_ROOT)
print("output_dir:", OUTPUT_DIR)

## Run Discovery

This table should answer: which configs exist, which checkpoints exist, and which generated sample files are available. The notebook supports the old `.npy` samples and the v2 `.npz` samples.

In [ ]:
def latest_checkpoint(run_name: str) -> Path | None:
    ckpt_dir = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    if not ckpt_dir.exists():
        return None
    candidates = sorted(ckpt_dir.glob("checkpoint-epoch-*"))
    return candidates[-1] if candidates else None


def checkpoint_epoch(path: Path | None) -> int | None:
    if path is None:
        return None
    m = re.search(r"checkpoint-epoch-(\d+)$", path.name)
    return int(m.group(1)) if m else None


def sample_candidates(run_name: str, ema_label: str | None = None) -> list[Path]:
    labels = [ema_label] if ema_label else EMA_LABELS
    labels = [x for x in labels if x]
    paths: list[Path] = []
    for label in labels:
        if PREFERRED_SAMPLER:
            paths.append(SAMPLE_ROOT / f"{run_name}_seed{SEED}_{label}_{PREFERRED_SAMPLER}.npz")
        paths.append(SAMPLE_ROOT / f"{run_name}_seed{SEED}_{label}.npy")
        paths.extend(sorted(SAMPLE_ROOT.glob(f"{run_name}_seed{SEED}_{label}_*.npz")))
    paths.extend(sorted(SAMPLE_ROOT.glob(f"{run_name}_seed{SEED}_*.npy")))
    paths.extend(sorted(SAMPLE_ROOT.glob(f"{run_name}_seed{SEED}_*.npz")))
    out = []
    seen = set()
    for p in paths:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def sample_shape(path: Path) -> tuple[int, ...] | None:
    if not path.exists():
        return None
    try:
        if path.suffix == ".npz":
            with np.load(path, allow_pickle=True) as data:
                key = "samples" if "samples" in data.files else data.files[0]
                return tuple(data[key].shape)
        return tuple(np.load(path, mmap_mode="r").shape)
    except Exception:
        return None


def parse_sample_label(path: Path) -> str:
    stem = path.stem
    for label in EMA_LABELS:
        if f"_{label}" in stem:
            return label
    return "unknown"


def parse_sampler_label(path: Path) -> str:
    stem = path.stem
    for label in EMA_LABELS:
        token = f"_{label}_"
        if token in stem:
            return stem.split(token, 1)[1]
    return "train_full" if path.suffix == ".npz" else "legacy_full"


if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Run the sweep config prep script first."
    )

manifest = json.loads(MANIFEST_PATH.read_text())
run_rows = []
for row in manifest:
    run_name = row["run_name"]
    ckpt = latest_checkpoint(run_name)
    config_path = PROJECT_DIR / row["config"]
    sample_exists = {label: any(p.exists() for p in sample_candidates(run_name, label)) for label in EMA_LABELS}
    first_sample = next((p for p in sample_candidates(run_name) if p.exists()), None)
    run_rows.append({
        **row,
        "config_path": str(config_path),
        "config_exists": config_path.exists(),
        "checkpoint_path": str(ckpt) if ckpt else None,
        "checkpoint_epoch": checkpoint_epoch(ckpt),
        "has_any_sample": first_sample is not None,
        "first_sample": str(first_sample) if first_sample else None,
        "first_sample_shape": sample_shape(first_sample) if first_sample else None,
        **{f"has_{label}": exists for label, exists in sample_exists.items()},
    })

run_df = pd.DataFrame(run_rows)
display_cols = [c for c in [
    "run_name", "arch", "variant_tag", "beta_schedule", "prediction_type",
    "sigma_log_normal", "min_snr_gamma", "config_exists", "checkpoint_epoch",
    "has_any_sample", "first_sample_shape", *[f"has_{label}" for label in EMA_LABELS],
] if c in run_df.columns]
display(run_df[display_cols])

## Optional One-Sample Smoke Generation

If discovery shows checkpoints but no generated samples, run this cell once with `GENERATE_ONE_SAMPLE_NOW = True`. It writes exactly one raw generated sample for the first run with a checkpoint. Then rerun the load cell below.

In [ ]:
GENERATE_ONE_SAMPLE_NOW = False  # change to True, run this cell, then rerun the load cell
SMOKE_DEVICE = "cuda"
SMOKE_NUM_SAMPLES = 1
SMOKE_BATCH_SIZE = 1
SMOKE_NUM_STEPS = None  # v2 only; set to 25 for a faster DPM smoke test if desired

if GENERATE_ONE_SAMPLE_NOW:
    import shlex
    import subprocess

    venv_python = Path("/home/jiamingp/venvs/cosmodiff_nf/bin/python")
    python_bin = str(venv_python if venv_python.exists() else Path(sys.executable))
    smoke_rows = [r for r in run_df.to_dict("records") if r.get("checkpoint_path")]
    if not smoke_rows:
        raise RuntimeError("No checkpoint_path found. Wait for a training job to write a checkpoint first.")

    row = smoke_rows[0]
    run_name = row["run_name"]
    SAMPLE_ROOT.mkdir(parents=True, exist_ok=True)
    stub_root = CACHE_DIR / "python_stubs"
    (stub_root / "sklearn" / "metrics").mkdir(parents=True, exist_ok=True)
    (stub_root / "sklearn" / "__init__.py").write_text("from . import metrics\n")
    (stub_root / "sklearn" / "metrics" / "__init__.py").write_text(
        "def roc_curve(*args, **kwargs):\n"
        "    raise RuntimeError('sklearn.metrics.roc_curve is stubbed for cosmodiff sampling')\n"
    )

    env = os.environ.copy()
    env["TORCHDYNAMO_DISABLE"] = "1"
    env["COSMODIFF_DIR"] = str(COSMODIFF_DIR)
    env["COSMODIFF_STUB_SKLEARN"] = "1"
    env.pop("PYTHONNOUSERSITE", None)
    env["PYTHONPATH"] = f"{stub_root}:{COSMODIFF_DIR}:{PROJECT_DIR}:{env.get('PYTHONPATH', '')}"

    if SWEEP_NAME == "nf_sweep_v2":
        output_path = SAMPLE_ROOT / f"{run_name}_seed{SEED}_raw_train_full.npz"
        cmd = [
            python_bin,
            str(COSMODIFF_DIR / "scripts" / "cosmodiff_sample.py"),
            "--config", str(row["config_path"]),
            "--filepath", str(output_path),
            "--n_samples", str(SMOKE_NUM_SAMPLES),
            "--batch_size", str(SMOKE_BATCH_SIZE),
            "--seed", str(SEED),
            "--device", SMOKE_DEVICE,
            "--verbose",
        ]
        if SMOKE_NUM_STEPS is not None:
            cmd += ["--scheduler", "DPMSolverMultistepScheduler", "--num_steps", str(SMOKE_NUM_STEPS)]
    else:
        output_path = SAMPLE_ROOT / f"{run_name}_seed{SEED}_raw.npy"
        cmd = [
            python_bin,
            str(PROJECT_DIR / "scripts" / "sample_cosmodiff.py"),
            "--checkpoint", str(row["checkpoint_path"]),
            "--config", str(row["config_path"]),
            "--output", str(output_path),
            "--num-samples", str(SMOKE_NUM_SAMPLES),
            "--batch-size", str(SMOKE_BATCH_SIZE),
            "--image-size", "128",
            "--seed", str(SEED),
            "--device", SMOKE_DEVICE,
        ]

    print("Generating one smoke sample for", run_name)
    print("Output:", output_path)
    print("Command:", " ".join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, env=env, check=True)
    print("Done. Rerun the Run Discovery cell and then the Load Real And Generated Arrays cell.")
else:
    print("Set GENERATE_ONE_SAMPLE_NOW = True to generate one raw smoke sample from the first available checkpoint.")

## Load Real And Generated Arrays

This is deliberately conservative. We load a limited real reference once per unique data config, then reuse it for all matching runs. That avoids killing the kernel by loading the same 128-grid real data repeatedly.

In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return np.array(arr[idx], copy=True)


def load_sample_array(path: Path) -> np.ndarray:
    if path.suffix == ".npz":
        with np.load(path, allow_pickle=True) as data:
            key = "samples" if "samples" in data.files else data.files[0]
            return as_nchw(np.asarray(data[key])).copy()
    return as_nchw(np.asarray(np.load(path, mmap_mode="r"))).copy()


def first_existing_sample(run_name: str, preferred: str = PREFERRED_SAMPLE_LABEL) -> tuple[str | None, str | None, Path | None]:
    order = [preferred] + [x for x in EMA_LABELS if x != preferred]
    for label in order:
        for path in sample_candidates(run_name, label):
            if path.exists():
                return parse_sample_label(path), parse_sampler_label(path), path
    return None, None, None


def data_signature(config_path: Path) -> str:
    with config_path.open() as f:
        cfg = yaml.safe_load(f)
    data = cfg.get("data", {})
    keys = ["img_path", "reshape", "two_dim", "zthin", "n_samples", "seed", "log", "transform", "normalization", "norm_kwargs"]
    slim = {k: data.get(k) for k in keys if k in data}
    return json.dumps(slim, sort_keys=True, default=str)


real_cache: dict[str, np.ndarray] = {}
loaded: dict[str, dict[str, Any]] = {}
load_rows = []
for row in run_df.to_dict("records"):
    run_name = row["run_name"]
    label, sampler, spath = first_existing_sample(run_name)
    config_path = Path(row["config_path"])
    if spath is None:
        load_rows.append({"run_name": run_name, "loaded": False, "reason": "missing generated sample"})
        continue
    if not config_path.exists():
        load_rows.append({"run_name": run_name, "loaded": False, "reason": "missing config"})
        continue

    sig = data_signature(config_path)
    if sig not in real_cache:
        real_cache[sig] = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
    real = real_cache[sig]
    generated = load_sample_array(spath)
    loaded[run_name] = {
        "spec": row,
        "sample_label": label,
        "sampler_label": sampler,
        "sample_path": spath,
        "real": real,
        "generated": generated,
    }
    load_rows.append({
        "run_name": run_name,
        "arch": row.get("arch"),
        "variant": row.get("variant_tag"),
        "loaded": True,
        "sample_label": label,
        "sampler": sampler,
        "real_shape": tuple(real.shape),
        "generated_shape": tuple(generated.shape),
        "sample_path": str(spath),
    })

load_df = pd.DataFrame(load_rows)
display(load_df)
print("loaded runs:", len(loaded), "unique real references:", len(real_cache))
if not loaded:
    print("No generated samples loaded yet. Submit/finish sampling before running the diagnostics below.")

## One-Point Statistics

Start here. This checks the pixel/field-value distribution: mean, width, tails, saturation near ±1, and histogram mismatch. If this fails badly, P(k) and encoder metrics are usually not meaningful yet.

In [ ]:
def onepoint_summary(real: np.ndarray, generated: np.ndarray, bins: int = 120) -> dict[str, float]:
    rh = field_histogram(real, bins=bins)
    gh = field_histogram(generated, bins=bins)
    edges = np.asarray(rh["bin_edges"])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh["hist"]) - np.asarray(gh["hist"]))) * width)
    return {
        "real_mean": rh["mean"],
        "generated_mean": gh["mean"],
        "mean_diff": gh["mean"] - rh["mean"],
        "real_std": rh["std"],
        "generated_std": gh["std"],
        "std_ratio": gh["std"] / max(rh["std"], 1e-30),
        "real_q01": rh["q01"],
        "generated_q01": gh["q01"],
        "real_q99": rh["q99"],
        "generated_q99": gh["q99"],
        "generated_frac_abs_ge_0999": gh["frac_abs_ge_0999"],
        "hist_l1": hist_l1,
    }

onepoint_rows = []
for run_name, bundle in loaded.items():
    real = evenly_limit(bundle["real"], MAX_REAL_HIST)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_HIST)
    onepoint_rows.append({
        "run_name": run_name,
        "arch": bundle["spec"].get("arch"),
        "variant": bundle["spec"].get("variant_tag"),
        "sample_label": bundle["sample_label"],
        "sampler": bundle["sampler_label"],
        **onepoint_summary(real, generated),
    })
onepoint_df = pd.DataFrame(onepoint_rows)
if len(onepoint_df):
    onepoint_df = onepoint_df.sort_values(["arch", "hist_l1"], na_position="last")
display(onepoint_df)
onepoint_csv = OUTPUT_DIR / f"{SWEEP_NAME}_onepoint_metrics.csv"
onepoint_df.to_csv(onepoint_csv, index=False)
print("wrote", onepoint_csv)

In [ ]:
if loaded:
    for arch in sorted({b["spec"].get("arch") for b in loaded.values()}):
        items = [(k, v) for k, v in loaded.items() if v["spec"].get("arch") == arch]
        ncols = min(4, len(items))
        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (run_name, bundle) in zip(axes.ravel(), items):
            ax.axis("on")
            real = evenly_limit(bundle["real"], MAX_REAL_HIST)
            generated = evenly_limit(bundle["generated"], MAX_GENERATED_HIST)
            real_hist = field_histogram(real)
            gen_hist = field_histogram(generated)
            edges = np.asarray(real_hist["bin_edges"])
            centers = 0.5 * (edges[:-1] + edges[1:])
            ax.plot(centers, real_hist["hist"], color="black", lw=2, label="real")
            ax.plot(centers, gen_hist["hist"], color="tab:blue", lw=1.8, label="generated")
            ax.set_yscale("log")
            ax.set_title(bundle["spec"].get("variant_tag", run_name), fontsize=10)
            ax.set_xlabel("normalized field value")
            ax.set_ylabel("density")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8)
        fig.suptitle(f"{arch}: one-point histograms")
        fig.tight_layout()
        out = OUTPUT_DIR / f"{SWEEP_NAME}_{arch}_onepoint_histograms.png"
        fig.savefig(out)
        print("wrote", out)

## Power Spectrum P(k)

This is the next standard check. The plot uses one real reference mean with a real ±1σ band. Generated curves are shown as a mean with a generated ±1σ band, and the lower panel is generated mean / real mean.

In [ ]:
pk_rows = []
pk_cache: dict[str, dict[str, np.ndarray]] = {}
for run_name, bundle in loaded.items():
    real = evenly_limit(bundle["real"], MAX_REAL_PK)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PK)
    pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
    pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
    real_mean = np.nanmean(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    ratio = gen_mean / np.clip(real_mean, 1e-30, None)
    pk_cache[run_name] = {"kbins": kbins, "pk_real": pk_real, "pk_gen": pk_gen, "ratio": ratio}
    pk_rows.append({
        "run_name": run_name,
        "arch": bundle["spec"].get("arch"),
        "variant": bundle["spec"].get("variant_tag"),
        "sample_label": bundle["sample_label"],
        "sampler": bundle["sampler_label"],
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })
pk_df = pd.DataFrame(pk_rows)
if len(pk_df):
    pk_df = pk_df.sort_values(["arch", "pk_log10_mae"], na_position="last")
display(pk_df)
pk_csv = OUTPUT_DIR / f"{SWEEP_NAME}_pk_metrics.csv"
pk_df.to_csv(pk_csv, index=False)
print("wrote", pk_csv)

In [ ]:
def plot_pk_detail(run_name: str, bundle: dict[str, Any]) -> None:
    info = pk_cache[run_name]
    kbins = info["kbins"]
    pk_real = info["pk_real"]
    pk_gen = info["pk_gen"]
    real_mean = np.nanmean(pk_real, axis=0)
    real_std = np.nanstd(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    gen_std = np.nanstd(pk_gen, axis=0)
    ratio = gen_mean / np.clip(real_mean, 1e-30, None)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ax = axes[0]
    ax.fill_between(kbins, np.clip(real_mean - real_std, 1e-30, None), real_mean + real_std, color="black", alpha=0.14, label="real ±1σ")
    ax.plot(kbins, real_mean, color="black", lw=2.2, label="real mean")
    ax.fill_between(kbins, np.clip(gen_mean - gen_std, 1e-30, None), gen_mean + gen_std, color="tab:blue", alpha=0.18, label="generated ±1σ")
    ax.plot(kbins, gen_mean, color="tab:blue", lw=2.0, label="generated mean")
    ax.set_yscale("log")
    ax.set_xlabel("k bin")
    ax.set_ylabel("P(k)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.plot(kbins, ratio, marker="o", color="tab:blue")
    ax.axhline(1.0, color="black", linestyle=":", linewidth=1.5)
    ax.set_xlabel("k bin")
    ax.set_ylabel("generated mean / real mean")
    ax.grid(alpha=0.25)
    ax.set_ylim(0, max(2.0, np.nanmax(ratio) * 1.1))

    title = f"{bundle['spec'].get('arch')} {bundle['spec'].get('variant_tag')} ({bundle['sample_label']}, {bundle['sampler_label']})"
    fig.suptitle(title)
    fig.tight_layout()
    out = OUTPUT_DIR / f"{SWEEP_NAME}_{run_name}_{bundle['sample_label']}_{bundle['sampler_label']}_pk.png"
    fig.savefig(out)
    print("wrote", out)

for run_name, bundle in loaded.items():
    plot_pk_detail(run_name, bundle)

## Real Versus Generated Images

Use this after the histogram/P(k) tables. It is a quick visual sanity check: real samples are the first row, generated samples are the second row.

In [ ]:
def image_grid_pair(real: np.ndarray, generated: np.ndarray, title: str, n: int = 8, cmap: str = "viridis") -> None:
    real = evenly_limit(as_nchw(real), min(n, len(real)))
    generated = evenly_limit(as_nchw(generated), min(n, len(generated)))
    n = min(len(real), len(generated), n)
    if n == 0:
        return
    fig, axes = plt.subplots(2, n, figsize=(1.7 * n, 3.4), squeeze=False)
    vals = np.concatenate([real[:n].ravel(), generated[:n].ravel()])
    vmin, vmax = np.nanpercentile(vals, [1, 99])
    for j in range(n):
        axes[0, j].imshow(real[j, 0], origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
        axes[1, j].imshow(generated[j, 0], origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
        axes[0, j].axis("off")
        axes[1, j].axis("off")
    axes[0, 0].set_ylabel("real")
    axes[1, 0].set_ylabel("generated")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

for run_name, bundle in loaded.items():
    image_grid_pair(
        evenly_limit(bundle["real"], MAX_REAL_IMAGES),
        evenly_limit(bundle["generated"], MAX_GENERATED_IMAGES),
        f"{run_name} ({bundle['sample_label']}, {bundle['sampler_label']})",
        n=min(MAX_REAL_IMAGES, MAX_GENERATED_IMAGES),
    )

## PCA Encoder

This replaces SSCD with a domain-local encoder. The encoder is fit on real training/reference slices only, using flattened normalized fields. Then real and generated samples are projected into PCA space.

The PCA generalizability score below is:

`1 - fraction(generated samples whose nearest real neighbor is above the real-real 99% similarity threshold)`

So higher is less copy-like under this PCA embedding. The absolute threshold is not a universal number; it is calibrated from the real reference used in this notebook.

In [ ]:
def flatten_images(images: np.ndarray) -> np.ndarray:
    arr = as_nchw(images).astype(np.float32, copy=False)
    return arr.reshape(len(arr), -1)


class PCAEncoder:
    def __init__(self, mean: np.ndarray, scale: np.ndarray, components: np.ndarray, explained_variance_ratio: np.ndarray):
        self.mean = mean.astype(np.float32)
        self.scale = scale.astype(np.float32)
        self.components = components.astype(np.float32)
        self.explained_variance_ratio = explained_variance_ratio.astype(np.float32)

    def transform(self, images: np.ndarray) -> np.ndarray:
        x = flatten_images(images)
        x = (x - self.mean) / self.scale
        return x @ self.components.T


def fit_pca_encoder(real: np.ndarray, n_components: int = 32, max_fit: int = 512) -> PCAEncoder:
    x = flatten_images(evenly_limit(real, max_fit))
    mean = x.mean(axis=0, keepdims=True)
    scale = x.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    x = (x - mean) / scale
    x = x.astype(np.float32, copy=False)
    n_components = int(min(n_components, x.shape[0] - 1, x.shape[1]))
    if n_components < 2:
        raise ValueError("Need at least 3 real samples to fit PCA.")

    try:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=n_components, svd_solver="randomized", random_state=0)
        pca.fit(x)
        components = pca.components_
        evr = pca.explained_variance_ratio_
    except Exception as exc:
        print("sklearn PCA unavailable or failed; using numpy SVD:", repr(exc))
        _, s, vt = np.linalg.svd(x, full_matrices=False)
        components = vt[:n_components]
        var = (s ** 2) / max(len(x) - 1, 1)
        evr = var[:n_components] / np.clip(var.sum(), 1e-30, None)
    return PCAEncoder(mean.squeeze(0), scale.squeeze(0), components, evr)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norm, 1e-12, None)


def pca_generalization_metrics(real_emb: np.ndarray, gen_emb: np.ndarray, quantile: float = 0.99) -> dict[str, float]:
    real_z = l2_normalize(real_emb)
    gen_z = l2_normalize(gen_emb)
    sim = gen_z @ real_z.T
    max_sim = sim.max(axis=1)

    rr = real_z @ real_z.T
    np.fill_diagonal(rr, -np.inf)
    real_nn = rr.max(axis=1)
    threshold = float(np.quantile(real_nn[np.isfinite(real_nn)], quantile))
    copy_fraction = float(np.mean(max_sim >= threshold))
    return {
        "pca_copy_threshold": threshold,
        "pca_copy_fraction": copy_fraction,
        "pca_generalization_score": 1.0 - copy_fraction,
        "pca_max_sim_median": float(np.median(max_sim)),
        "pca_max_sim_q95": float(np.quantile(max_sim, 0.95)),
        "pca_max_sim_q99": float(np.quantile(max_sim, 0.99)),
    }

In [ ]:
pca_rows = []
pca_cache: dict[str, dict[str, np.ndarray]] = {}
encoder_cache: dict[str, PCAEncoder] = {}
for run_name, bundle in loaded.items():
    # Data signature makes the PCA basis shared across runs using the same real data/normalization.
    sig = data_signature(Path(bundle["spec"]["config_path"]))
    if sig not in encoder_cache:
        encoder_cache[sig] = fit_pca_encoder(bundle["real"], n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
    encoder = encoder_cache[sig]

    real = evenly_limit(bundle["real"], PCA_MAX_REAL)
    generated = evenly_limit(bundle["generated"], PCA_MAX_GENERATED)
    real_emb = encoder.transform(real)
    gen_emb = encoder.transform(generated)
    pca_cache[run_name] = {"real_emb": real_emb, "gen_emb": gen_emb, "explained_variance_ratio": encoder.explained_variance_ratio}
    pca_rows.append({
        "run_name": run_name,
        "arch": bundle["spec"].get("arch"),
        "variant": bundle["spec"].get("variant_tag"),
        "sample_label": bundle["sample_label"],
        "sampler": bundle["sampler_label"],
        "pca_components": len(encoder.explained_variance_ratio),
        "pca_explained_variance_sum": float(encoder.explained_variance_ratio.sum()),
        **pca_generalization_metrics(real_emb, gen_emb, quantile=PCA_COPY_QUANTILE),
    })
pca_df = pd.DataFrame(pca_rows)
if len(pca_df):
    pca_df = pca_df.sort_values(["arch", "pca_generalization_score"], ascending=[True, False], na_position="last")
display(pca_df)
pca_csv = OUTPUT_DIR / f"{SWEEP_NAME}_pca_generalization_metrics.csv"
pca_df.to_csv(pca_csv, index=False)
print("wrote", pca_csv)

In [ ]:
# PCA explained variance: if all runs share the same real data, this will be one curve.
if encoder_cache:
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, encoder in enumerate(encoder_cache.values()):
        y = np.cumsum(encoder.explained_variance_ratio)
        ax.plot(np.arange(1, len(y) + 1), y, marker="o", label=f"encoder {i}")
    ax.set_xlabel("PCA component")
    ax.set_ylabel("cumulative explained variance")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    out = OUTPUT_DIR / f"{SWEEP_NAME}_pca_explained_variance.png"
    fig.savefig(out)
    print("wrote", out)

In [ ]:
# PCA scatter: real gray cloud versus generated blue cloud for each run.
if pca_cache:
    for arch in sorted({b["spec"].get("arch") for b in loaded.values()}):
        items = [(k, v) for k, v in loaded.items() if v["spec"].get("arch") == arch]
        ncols = min(4, len(items))
        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.4 * nrows), squeeze=False)
        for ax in axes.ravel():
            ax.axis("off")
        for ax, (run_name, bundle) in zip(axes.ravel(), items):
            ax.axis("on")
            real_emb = pca_cache[run_name]["real_emb"]
            gen_emb = pca_cache[run_name]["gen_emb"]
            ax.scatter(real_emb[:, 0], real_emb[:, 1], s=8, c="black", alpha=0.18, label="real")
            ax.scatter(gen_emb[:, 0], gen_emb[:, 1], s=14, c="tab:blue", alpha=0.6, label="generated")
            ax.set_title(bundle["spec"].get("variant_tag", run_name), fontsize=10)
            ax.set_xlabel("PC1")
            ax.set_ylabel("PC2")
            ax.grid(alpha=0.2)
        handles, labels = axes.ravel()[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper right")
        fig.suptitle(f"{arch}: PCA encoder space")
        fig.tight_layout()
        out = OUTPUT_DIR / f"{SWEEP_NAME}_{arch}_pca_scatter.png"
        fig.savefig(out)
        print("wrote", out)

In [ ]:
# PCA generalizability plot. For nf_sweep all runs have n=500, so compare variants.
if len(pca_df):
    for arch, sub in pca_df.groupby("arch"):
        sub = sub.sort_values("pca_generalization_score", ascending=False)
        fig, ax = plt.subplots(figsize=(max(7, 0.6 * len(sub)), 4.2))
        ax.bar(sub["variant"], sub["pca_generalization_score"], color="tab:green", alpha=0.75)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("PCA generalization score")
        ax.set_xlabel("variant")
        ax.set_title(f"{arch}: PCA-based generalizability ({PCA_COPY_QUANTILE:.0%} real-real threshold)")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        out = OUTPUT_DIR / f"{SWEEP_NAME}_{arch}_pca_generalizability.png"
        fig.savefig(out)
        print("wrote", out)

## EMA Target Sweep Metrics

After the sampling array finishes, this compares all raw/EMA samples that exist. The first metric to look at here is `pk_log10_mae`; lower means the generated mean P(k) is closer to the real mean P(k).

In [ ]:
ema_metric_rows = []
for row in run_df.to_dict("records"):
    run_name = row["run_name"]
    config_path = Path(row["config_path"])
    if not config_path.exists():
        continue
    sig = data_signature(config_path)
    real = real_cache.get(sig)
    if real is None:
        continue
    for label in EMA_LABELS:
        for path in sample_candidates(run_name, label):
            if not path.exists():
                continue
            generated = load_sample_array(path)
            onepoint = onepoint_summary(evenly_limit(real, MAX_REAL_HIST), evenly_limit(generated, MAX_GENERATED_HIST))
            pk_summary = power_spectrum_summary(evenly_limit(real, MAX_REAL_PK), evenly_limit(generated, MAX_GENERATED_PK), nbins=PK_NBINS)
            ema_metric_rows.append({
                "run_name": run_name,
                "arch": row.get("arch"),
                "variant": row.get("variant_tag"),
                "ema_label": parse_sample_label(path),
                "ema_value": EMA_VALUES.get(parse_sample_label(path), np.nan),
                "sampler": parse_sampler_label(path),
                "sample_path": str(path),
                "hist_l1": onepoint["hist_l1"],
                "generated_std": onepoint["generated_std"],
                "generated_frac_abs_ge_0999": onepoint["generated_frac_abs_ge_0999"],
                **pk_summary,
            })
            break
ema_metric_df = pd.DataFrame(ema_metric_rows)
if len(ema_metric_df):
    ema_metric_df = ema_metric_df.sort_values(["arch", "variant", "sampler", "pk_log10_mae"], na_position="last")
display(ema_metric_df)
ema_metric_csv = OUTPUT_DIR / f"{SWEEP_NAME}_ema_target_metrics.csv"
ema_metric_df.to_csv(ema_metric_csv, index=False)
print("wrote", ema_metric_csv)

In [ ]:
if len(ema_metric_df):
    for arch, sub_arch in ema_metric_df.groupby("arch"):
        # Keep the plot readable: one sampler at a time.
        for sampler, sub in sub_arch.groupby("sampler"):
            fig, ax = plt.subplots(figsize=(8, 5))
            for variant, sub_v in sub.groupby("variant"):
                sub_v = sub_v.sort_values("ema_value", na_position="first")
                x = sub_v["ema_value"].fillna(-0.01)
                ax.plot(x, sub_v["pk_log10_mae"], marker="o", label=variant)
            ax.set_xlabel("EMA sigma_rel (-0.01 = raw)")
            ax.set_ylabel("P(k) log10 MAE, lower is better")
            ax.set_title(f"{arch}: EMA target sweep ({sampler})")
            ax.grid(alpha=0.25)
            ax.legend(fontsize=8, ncol=2)
            fig.tight_layout()
            out = OUTPUT_DIR / f"{SWEEP_NAME}_{arch}_{sampler}_ema_pk_sweep.png"
            fig.savefig(out)
            print("wrote", out)

## Training Curves

Training curves are useful, but they do not decide sample quality by themselves. Use them after the one-point/P(k)/PCA checks.

In [ ]:
def metric_candidates(run_name: str) -> list[Path]:
    root = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    paths = []
    paths.extend(sorted(root.glob("metrics_epoch_*.json")))
    paths.extend(sorted(root.glob("metrics.json")))
    for ckpt in sorted(root.glob("checkpoint-epoch-*")):
        paths.extend(sorted(ckpt.glob("metrics*.json")))
    return paths


def read_metrics_file(path: Path) -> dict[str, Any]:
    with path.open() as f:
        return json.load(f)


def latest_metrics(run_name: str) -> tuple[Path | None, dict[str, Any] | None]:
    candidates = metric_candidates(run_name)
    if not candidates:
        return None, None
    path = candidates[-1]
    return path, read_metrics_file(path)

metrics_rows = []
metrics_by_run = {}
for row in run_df.to_dict("records"):
    run_name = row["run_name"]
    path, metrics = latest_metrics(run_name)
    if metrics is None:
        metrics_rows.append({"run_name": run_name, "arch": row.get("arch"), "variant": row.get("variant_tag"), "has_metrics": False})
        continue
    metrics_by_run[run_name] = metrics
    epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
    batch_loss = np.asarray(metrics.get("loss", metrics.get("batch_loss", [])), dtype=float)
    lr = np.asarray(metrics.get("lr", metrics.get("learning_rate", [])), dtype=float)
    metrics_rows.append({
        "run_name": run_name,
        "arch": row.get("arch"),
        "variant": row.get("variant_tag"),
        "has_metrics": True,
        "metrics_path": str(path),
        "n_epochs_logged": len(epoch_loss),
        "latest_epoch_loss": float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        "best_epoch_loss": float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
        "n_batch_loss": len(batch_loss),
        "n_lr": len(lr),
    })
metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

In [ ]:
def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    if len(x) == 0 or window <= 1:
        return x
    window = min(window, len(x))
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")

for arch in sorted(metrics_df.get("arch", pd.Series(dtype=str)).dropna().unique()):
    sub = metrics_df[(metrics_df["arch"] == arch) & (metrics_df["has_metrics"])]
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for _, row in sub.iterrows():
        metrics = metrics_by_run.get(row["run_name"], {})
        label = row["variant"]
        batch_loss = np.asarray(metrics.get("loss", metrics.get("batch_loss", [])), dtype=float)
        epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
        lr = np.asarray(metrics.get("lr", metrics.get("learning_rate", [])), dtype=float)
        if len(batch_loss):
            y = moving_average(batch_loss, max(1, len(batch_loss) // 400))
            axes[0].plot(np.arange(len(y)), y, lw=1.2, label=label)
        if len(epoch_loss):
            axes[1].plot(np.arange(len(epoch_loss)), epoch_loss, marker="o", ms=2.5, lw=1.2, label=label)
        if len(lr):
            axes[2].plot(np.arange(len(lr)), lr, lw=1.0, label=label)
    axes[0].set_title("batch loss")
    axes[0].set_xlabel("optimizer step")
    axes[0].set_ylabel("MSE loss")
    axes[1].set_title("epoch loss")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mean MSE loss")
    axes[2].set_title("learning rate")
    axes[2].set_xlabel("logged step/epoch")
    axes[2].set_ylabel("LR")
    axes[2].set_yscale("log")
    for ax in axes:
        ax.grid(alpha=0.25)
    axes[1].legend(fontsize=8, ncol=2)
    fig.suptitle(f"{arch}: training curves")
    fig.tight_layout()
    out = OUTPUT_DIR / f"{SWEEP_NAME}_{arch}_training_curves.png"
    fig.savefig(out)
    print("wrote", out)

## Final Ranking Table

Use this only after the diagnostics above make sense. A decent model should have low histogram mismatch, low P(k) mismatch, and high PCA generalization score.

In [ ]:
summary = run_df[[c for c in ["run_name", "arch", "variant_tag", "beta_schedule", "prediction_type", "sigma_log_normal", "min_snr_gamma", "checkpoint_epoch", "has_any_sample"] if c in run_df.columns]].copy()
summary = summary.rename(columns={"variant_tag": "variant"})
for df, cols in [
    (onepoint_df, ["run_name", "sample_label", "sampler", "hist_l1", "std_ratio", "generated_frac_abs_ge_0999"]),
    (pk_df, ["run_name", "pk_log10_mae", "pk_ratio_low_k", "pk_ratio_mid_k", "pk_ratio_high_k"]),
    (pca_df, ["run_name", "pca_generalization_score", "pca_copy_fraction", "pca_max_sim_median", "pca_copy_threshold"]),
    (metrics_df, ["run_name", "n_epochs_logged", "latest_epoch_loss", "best_epoch_loss"]),
]:
    if len(df):
        summary = summary.merge(df[[c for c in cols if c in df.columns]], on="run_name", how="left")

sort_cols = [c for c in ["arch", "pk_log10_mae", "hist_l1"] if c in summary.columns]
if sort_cols:
    summary = summary.sort_values(sort_cols, na_position="last")
display(summary)
summary_csv = OUTPUT_DIR / f"{SWEEP_NAME}_summary.csv"
summary.to_csv(summary_csv, index=False)
print("wrote", summary_csv)